<a href="https://colab.research.google.com/github/muhendis/langchain-langgraph-llm-uygulamalari/blob/master/part4/02_build_rag_search_agent_with_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook explores the development of advanced Language Model (LLM) applications. Specifically, we will dive into building intelligent agents using **LangChain** and integrating powerful **Google Gemini models**.

You'll learn how to combine **Retrieval Augmented Generation (RAG)** for accessing internal knowledge, **tool usage** for interacting with external services like web search, and agentic reasoning to create robust and capable LLM systems.

Key areas covered include:

*   **Environment Setup**: Installing necessary libraries and configuring API keys.
*   **Knowledge Base Construction**: Building a RAG system from external documents.
*   **Tool Definition**: Creating and integrating tools for both internal knowledge retrieval and external web searches.
*   **Agent Development**: Designing an LLM agent that intelligently orchestrates these tools based on a prioritized decision-making process.
*   **Testing**: Demonstrating the agent's ability to handle queries requiring both internal knowledge and external information.

By the end of this notebook, you'll have a clear understanding of how these components interoperate to create dynamic and effective LLM-powered solutions.

# Environment Setup

In [3]:
!pip install -U "langchain[google-genai]" langchain_chroma langchain_community
!pip install rich

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/5

In [4]:
import os
os.environ["GOOGLE_API_KEY"] = "Your-key"
os.environ["TAVILY_API_KEY"] = "Your-key" #https://app.tavily.com/home


In [6]:
import bs4
import os

# LangChain and LangGraph components
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import  create_agent
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.chat_models import init_chat_model

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 1. Knowledge Base Construction
Building a RAG system from external documents.



In [7]:


# Load blog content
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

# Create Vector Store (using Chroma)
vector_store = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings,
    collection_name="agent_blog_context"
)



# 2. Tool Definition

 Creating and integrating tools for both internal knowledge retrieval and external web searches.


In [8]:
# Define the Retrieval Tool
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """
    Searches the internal vector database to retrieve relevant documents
    and context matching the input query.
    """
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs


In [9]:

tavily_tool = TavilySearchResults(
    max_results=3,
    description=(
        "Performs a live web search to find current information, facts, "
        "and data from the internet."
    )
)


/tmp/ipython-input-3171867092.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(


# 3. Agent Development

Designing an LLM agent that intelligently orchestrates these tools based on a prioritized decision-making process.


In [10]:
# List of tools
tools = [retrieve_context, tavily_tool]

# Initialize Model
model = init_chat_model(
    "google_genai:gemini-2.5-flash",
)
# System Prompt
system_prompt = """
    You are a helpful assistant. You must answer the user's questions using the following strictly ordered process:
    1. **SEARCH INTERNAL**: First, use the 'retrieve_context' tool to check for local information regarding the query.
    2. **EVALUATE**: If the 'retrieve_context' output contains the answer, use it to respond and stop there.
    3. **SEARCH EXTERNAL**: ONLY if the internal context is missing or insufficient, use the 'tavily_search_results_json' tool to look up the answer on the web.
"""


# Create the Agent
agent = create_agent(model, tools, system_prompt=system_prompt)

# 4.Testing

 Demonstrating the agent's ability to handle queries requiring both internal knowledge and external information.


In [11]:
# Test 1: Question about the blog (Should use retrieve_context)

query = "What is 'Self-Reflection' in the context of LLM agents?"
for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is 'Self-Reflection' in the context of LLM agents?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (cb7cbd7a-4fdf-4e26-9b6d-0b61b328ccd1)
 Call ID: cb7cbd7a-4fdf-4e26-9b6d-0b61b328ccd1
  Args:
    query: Self-Reflection in LLM agents
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
Content: Self-Reflection#
Self-reflection is a vital aspect that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It plays a crucial role in real-world tasks where trial and error are inevitable.
ReAct (Yao et al. 2023) integrates reasoning and acting within LLM by extending the action space to be a combination of task-specific discrete actions and the language spac

In [12]:

# Test 2: Question about general knowledge (Should skip/fail RAG and use Tavily)
query ="What nation hosted the Euro 2024?"
for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What nation hosted the Euro 2024?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (7fd20863-368c-4b9f-821d-41b29428956f)
 Call ID: 7fd20863-368c-4b9f-821d-41b29428956f
  Args:
    query: Who hosted Euro 2024?
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
Content: Conversatin samples:
[
  {
    "role": "system",

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
Content: }
]
Then after these clarification, the agent moved into the code writing mode with a different system message.
System message:
================================== Ai Message ==================================
Tool Calls:
  tavily_search_results_json (deb66790-7b80-4178-861c-ed14e50b0d6f)
 Call ID: deb66790-7b80-4178-861c